# DupeLens
Runs the pipeline stage by stage (each stage caches its output, so re-running is cheap) and shows the results. Run from `notebooks/`.

In [ ]:
import os; os.chdir('..')  # repo root
import pandas as pd, json
pd.set_option('display.width', 200)

## 1. Tests and group split

In [ ]:
!"../.venv/bin/python" -m pytest tests -q -p no:cacheprovider
!"../.venv/bin/python" -m src.data

## 2. Embeddings (skips if cached)

In [ ]:
!"../.venv/bin/python" -m src.embed

## 3. PCA sweep and candidate recall (validation)

In [ ]:
!"../.venv/bin/python" -m src.candidates

## 4. Baselines and models (validation only)

In [ ]:
!"../.venv/bin/python" -m src.train

## 5. Single test run
Already done once; the script refuses a second run (`cache/test_done.json`). The test rows are shown below.

In [ ]:
r = pd.read_csv('results/results.csv')
r[r.split == 'test'][['step','model','features','dim','f1','f1_ci_low','f1_ci_high','pair_precision','pair_recall','cand_recall','threshold']]

## 6. India (frozen model)

In [ ]:
s = json.load(open('results/json/india_summary.json'))
{k: s[k] for k in ['category', 'shift_check', 'labelling', 'holdout']}

## 7. Plots and dashboard data

In [ ]:
!"../.venv/bin/python" -m src.export
from IPython.display import Image, display
for p in ['recall_vs_dims', 'pr_curve_test', 'explained_variance']:
    display(Image(f'results/plots/{p}.png'))